# 01 — Data Ingestion
**Bluestock Fintech MF Capstone**

In [1]:
import pandas as pd
import os
from pathlib import Path
import requests
import warnings
warnings.filterwarnings('ignore')

# Ensure CWD is project root (one level up from notebooks/)
PROJECT_ROOT = Path(__file__).resolve().parent.parent if '__file__' in dir() else Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")
print("Libraries imported successfully.")

Working directory: C:\Users\prati\.gemini\antigravity\scratch\bluestock_project
Libraries imported successfully.


In [2]:
RAW_DIR = Path("data/raw")

csv_files = sorted([f for f in RAW_DIR.glob("*.csv") if f.name.startswith(("01","02","03","04","05","06","07","08","09","10"))])

datasets = {}
for fp in csv_files:
    df = pd.read_csv(fp)
    datasets[fp.name] = df
    print(f"\n{'='*60}")
    print(f"File: {fp.name}")
    print(f"Shape: {df.shape}")
    print(f"Dtypes:\n{df.dtypes}")
    print(f"\nHead(3):")
    print(df.head(3))

print(f"\n\nTotal datasets loaded: {len(datasets)}")


File: 01_fund_master.csv
Shape: (40, 15)
Dtypes:
amfi_code               int64
fund_house                str
scheme_name               str
category                  str
sub_category              str
plan                      str
launch_date               str
benchmark                 str
expense_ratio_pct     float64
exit_load_pct         float64
min_sip_amount          int64
min_lumpsum_amount      int64
fund_manager              str
risk_category             str
sebi_category_code        str
dtype: object

Head(3):
   amfi_code       fund_house                                 scheme_name  \
0     119551  SBI Mutual Fund   SBI Bluechip Fund - Regular Plan - Growth   
1     119552  SBI Mutual Fund    SBI Bluechip Fund - Direct Plan - Growth   
2     119598  SBI Mutual Fund  SBI Small Cap Fund - Regular Plan - Growth   

  category sub_category     plan launch_date             benchmark  \
0   Equity    Large Cap  Regular  2006-02-14         NIFTY 100 TRI   
1   Equity    Large Cap   D

In [3]:
scheme_codes = [119551, 120503, 118632, 119092, 120841]

for code in scheme_codes:
    url = f"https://api.mfapi.in/mf/{code}"
    try:
        resp = requests.get(url, timeout=15)
        resp.raise_for_status()
        jdata = resp.json()
        scheme_name = jdata.get("meta", {}).get("scheme_name", "Unknown")
        nav_data = jdata.get("data", [])
        df_nav = pd.DataFrame(nav_data)
        print(f"\nScheme {code} — {scheme_name}")
        print(f"  Records fetched: {len(df_nav)}")
        print(df_nav.head(5))
    except Exception as e:
        print(f"\nScheme {code} — ERROR: {e}")


Scheme 119551 — Aditya Birla Sun Life Banking & PSU Debt Fund  - DIRECT - IDCW
  Records fetched: 3241
         date        nav
0  08-06-2026  105.30840
1  05-06-2026  105.10700
2  04-06-2026  104.80950
3  03-06-2026  104.74960
4  02-06-2026  104.76630



Scheme 120503 — Axis ELSS Tax Saver Fund - Direct Plan - Growth Option
  Records fetched: 3312
         date        nav
0  08-06-2026  102.48450
1  05-06-2026  103.81020
2  04-06-2026  103.64210
3  03-06-2026  103.48090
4  02-06-2026  103.59590



Scheme 118632 — Nippon India Large Cap Fund - Direct Plan Growth Plan - Growth Option
  Records fetched: 3303
         date       nav
0  08-06-2026  95.89520
1  05-06-2026  97.24560
2  04-06-2026  97.26210
3  03-06-2026  97.21670
4  02-06-2026  97.47830



Scheme 119092 — HDFC Money Market Fund - Growth Option - Direct Plan
  Records fetched: 3570
         date         nav
0  08-06-2026  6174.20840
1  05-06-2026  6168.47550
2  04-06-2026  6160.94340
3  03-06-2026  6159.96260
4  02-06-2026  6158.96650



Scheme 120841 — quant Mid Cap Fund - Growth Option - Direct Plan
  Records fetched: 3306
         date        nav
0  08-06-2026  242.15940
1  05-06-2026  246.08780
2  04-06-2026  246.07840
3  03-06-2026  245.65930
4  02-06-2026  246.85660


In [4]:
fund_master = pd.read_csv("data/raw/01_fund_master.csv")
nav_history = pd.read_csv("data/raw/02_nav_history.csv")

master_codes = set(fund_master["amfi_code"].unique())
nav_codes = set(nav_history["amfi_code"].unique())

matched = nav_codes & master_codes
unmatched = nav_codes - master_codes

print(f"AMFI codes in fund_master: {len(master_codes)}")
print(f"AMFI codes in nav_history: {len(nav_codes)}")
print(f"Matched codes: {len(matched)}")
print(f"Unmatched codes (in nav but not in master): {len(unmatched)}")
if unmatched:
    print(f"  Unmatched: {unmatched}")
else:
    print("  ✅ All nav_history codes exist in fund_master.")

AMFI codes in fund_master: 40
AMFI codes in nav_history: 40
Matched codes: 40
Unmatched codes (in nav but not in master): 0
  ✅ All nav_history codes exist in fund_master.


In [5]:
fm = pd.read_csv("data/raw/01_fund_master.csv")

print("=== Unique Fund Houses ===")
for fh in sorted(fm["fund_house"].unique()):
    print(f"  • {fh}")

print(f"\n=== Unique Categories ({fm['category'].nunique()}) ===")
for c in sorted(fm["category"].unique()):
    print(f"  • {c}")

print(f"\n=== Unique Sub-Categories ({fm['sub_category'].nunique()}) ===")
for sc in sorted(fm["sub_category"].unique()):
    print(f"  • {sc}")

print(f"\n=== Unique Risk Grades ===")
for rg in sorted(fm["risk_category"].unique()):
    print(f"  • {rg}")

=== Unique Fund Houses ===
  • Aditya Birla Sun Life MF
  • Axis Mutual Fund
  • DSP Mutual Fund
  • HDFC Mutual Fund
  • ICICI Prudential MF
  • Kotak Mahindra MF
  • Mirae Asset MF
  • Nippon India MF
  • SBI Mutual Fund
  • UTI Mutual Fund

=== Unique Categories (2) ===
  • Debt
  • Equity

=== Unique Sub-Categories (12) ===
  • ELSS
  • Flexi Cap
  • Gilt
  • Index
  • Index/ETF
  • Large & Mid Cap
  • Large Cap
  • Liquid
  • Mid Cap
  • Short Duration
  • Small Cap
  • Value

=== Unique Risk Grades ===
  • High
  • Low
  • Moderate
  • Moderately High
  • Very High


In [6]:
RAW_DIR = Path("data/raw")
csv_files = sorted([f for f in RAW_DIR.glob("*.csv") if f.name.startswith(("01","02","03","04","05","06","07","08","09","10"))])

print("=" * 70)
print("DATA QUALITY SUMMARY")
print("=" * 70)

for fp in csv_files:
    df = pd.read_csv(fp)
    null_total = df.isnull().sum().sum()
    dup_rows = df.duplicated().sum()
    print(f"\n{fp.name}  (rows={df.shape[0]}, cols={df.shape[1]})")
    print(f"  Null values : {null_total}")
    if null_total > 0:
        null_cols = df.isnull().sum()
        for col, cnt in null_cols[null_cols > 0].items():
            print(f"    - {col}: {cnt}")
    print(f"  Duplicate rows: {dup_rows}")

DATA QUALITY SUMMARY

01_fund_master.csv  (rows=40, cols=15)
  Null values : 0
  Duplicate rows: 0

02_nav_history.csv  (rows=46000, cols=3)
  Null values : 0
  Duplicate rows: 0

03_aum_by_fund_house.csv  (rows=90, cols=5)
  Null values : 0
  Duplicate rows: 0

04_monthly_sip_inflows.csv  (rows=48, cols=6)
  Null values : 12
    - yoy_growth_pct: 12
  Duplicate rows: 0

05_category_inflows.csv  (rows=144, cols=3)
  Null values : 0
  Duplicate rows: 0

06_industry_folio_count.csv  (rows=21, cols=6)
  Null values : 0
  Duplicate rows: 0

07_scheme_performance.csv  (rows=40, cols=19)
  Null values : 0
  Duplicate rows: 0



08_investor_transactions.csv  (rows=32778, cols=13)
  Null values : 0
  Duplicate rows: 0

09_portfolio_holdings.csv  (rows=322, cols=8)
  Null values : 0
  Duplicate rows: 0



10_benchmark_indices.csv  (rows=8050, cols=3)
  Null values : 0
  Duplicate rows: 0


## Summary
All 10 datasets loaded. Live NAV fetched for 5 schemes. Data quality assessed.